# Visualización de Trades Históricos

Este notebook permite visualizar las operaciones generadas por el sistema de trading sobre los datos de mercado históricos.

**Requisitos:**
Este notebook utiliza `plotly` para gráficos interactivos. Si no lo tienes instalado, ejecuta:
```bash
pip install plotly nbformat
```

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import os
import glob
import sys
import re
from datetime import timedelta
from pathlib import Path

# Configurar renderer para que funcione correctamente en VS Code
pio.renderers.default = "notebook_connected"

# Agregar src/ al path de Python para importar módulos del proyecto
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Importar función de cálculo de indicadores
from trading_strategy.indicators import calculate_indicator_and_signals

# Configuración de rutas relativas
TRADES_DIR = os.path.join('..', 'data', 'trades')
MARKET_DIR = os.path.join('..', 'data', 'market')

print(f"Directorio de Trades: {os.path.abspath(TRADES_DIR)}")
print(f"Directorio de Mercado: {os.path.abspath(MARKET_DIR)}")

Directorio de Trades: d:\py_projects\GammaNeutral\main\data\trades
Directorio de Mercado: d:\py_projects\GammaNeutral\main\data\market


In [2]:
def load_trades(filename):
    """Carga el archivo de trades y procesa las fechas."""
    path = os.path.join(TRADES_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró el archivo de trades: {path}")
    
    df = pd.read_csv(path)
    df['entry_time'] = pd.to_datetime(df['entry_time'])
    df['exit_time'] = pd.to_datetime(df['exit_time'])
    return df

def load_market_data(ticker, timeframe):
    """Busca y carga el archivo parquet correspondiente al ticker y timeframe."""
    pattern = os.path.join(MARKET_DIR, f"{ticker.lower()}_{timeframe.lower()}_*.parquet")
    files = glob.glob(pattern)
    
    if not files:
        raise FileNotFoundError(f"No se encontraron datos de mercado para {ticker} {timeframe}")
    
    file_path = files[0]
    print(f"Cargando datos de mercado desde: {file_path}")
    
    df = pd.read_parquet(file_path)
    
    rename_map = {
        'open': 'Open', 'high': 'High', 'low': 'Low', 
        'close': 'Close', 'volume': 'Volume'
    }
    df.rename(columns=rename_map, inplace=True)
    
    return df

def parse_strategy_from_filename(filename):
    """
    Intenta extraer el indicador y sus parámetros del nombre del archivo.
    Ejemplo: trades_UNIUSDT_1h_RSI_ob75os20p20_L.csv
    """
    parts = filename.replace('.csv', '').split('_')
    
    # Estructura esperada: trades, TICKER, TF, INDICATOR, PARAMS, TYPE
    if len(parts) < 5:
        return None, {}
        
    indicator = parts[3].lower()
    params_str = parts[4]
    
    params = {}
    
    # Parsing específico para RSI (ob, os, p)
    if indicator == 'rsi':
        ob_match = re.search(r'ob(\d+)', params_str)
        os_match = re.search(r'os(\d+)', params_str)
        p_match = re.search(r'p(\d+)', params_str)
        
        if ob_match: params['overbought'] = int(ob_match.group(1))
        if os_match: params['oversold'] = int(os_match.group(1))
        if p_match: params['period'] = int(p_match.group(1))
        
    # Parsing para MACD (f, s, sig) - Ej: f12s26sig9
    elif indicator == 'macd':
        f_match = re.search(r'f(\d+)', params_str)
        s_match = re.search(r's(\d+)', params_str)
        sig_match = re.search(r'sig(\d+)', params_str)
        
        if f_match: params['fast'] = int(f_match.group(1))
        if s_match: params['slow'] = int(s_match.group(1))
        if sig_match: params['signal'] = int(sig_match.group(1))
        
    return indicator, params

In [3]:
def plot_trades_on_chart(market_df, trades_df, indicator_name=None, indicator_params=None, title="Análisis de Trades"):
    """Genera un gráfico de velas con los trades superpuestos y el indicador debajo."""
    
    # 1. Calcular Indicador si se especifica
    df_plot = market_df.copy()
    if indicator_name:
        print(f"Calculando indicador: {indicator_name} con params {indicator_params}")
        # Usamos la función del framework para calcular el indicador
        # inplace=True modifica df_plot añadiendo columnas
        calculate_indicator_and_signals(df_plot, indicator_name, indicator_params, inplace=True)
    
    # 2. Normalizar Fechas
    if df_plot.index.tz is not None:
        df_plot.index = df_plot.index.tz_localize(None)
    if trades_df['entry_time'].dt.tz is not None:
        trades_df['entry_time'] = trades_df['entry_time'].dt.tz_localize(None)
    if trades_df['exit_time'].dt.tz is not None:
        trades_df['exit_time'] = trades_df['exit_time'].dt.tz_localize(None)

    # 3. Filtrar Rango
    start_date = trades_df['entry_time'].min() - timedelta(days=2)
    end_date = trades_df['exit_time'].max() + timedelta(days=2)
    
    mask = (df_plot.index >= start_date) & (df_plot.index <= end_date)
    df_view = df_plot.loc[mask]
    
    if df_view.empty:
        print("⚠️ No hay datos en el rango seleccionado.")
        return go.Figure()

    # 4. Crear Subplots (2 filas: Precio e Indicador)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.05, row_heights=[0.7, 0.3],
                        subplot_titles=('Precio y Trades', f'Indicador: {indicator_name.upper()}' if indicator_name else ''))

    # --- FILA 1: Precio y Trades ---
    fig.add_trace(go.Candlestick(
        x=df_view.index,
        open=df_view['Open'], high=df_view['High'],
        low=df_view['Low'], close=df_view['Close'],
        name='Precio'
    ), row=1, col=1)

    # Trades Markers
    long_entries = trades_df[trades_df['type'] == 'long']
    if not long_entries.empty:
        fig.add_trace(go.Scatter(
            x=long_entries['entry_time'], y=long_entries['entry_price'],
            mode='markers', marker=dict(symbol='triangle-up', size=12, color='green'),
            name='Long Entry', hovertemplate='Long Entry<br>Price: %{y:.2f}<extra></extra>'
        ), row=1, col=1)

    long_exits = trades_df[trades_df['type'] == 'long']
    if not long_exits.empty:
        fig.add_trace(go.Scatter(
            x=long_exits['exit_time'], y=long_exits['exit_price'],
            mode='markers', marker=dict(symbol='circle', size=10, color='red', line=dict(width=2, color='darkred')),
            name='Long Exit', text=long_exits['pnl_pct'],
            hovertemplate='Long Exit<br>Price: %{y:.2f}<br>PnL: %{text:.2%}<extra></extra>'
        ), row=1, col=1)

    # --- FILA 2: Indicador ---
    if indicator_name:
        if indicator_name == 'rsi':
            # RSI Line
            fig.add_trace(go.Scatter(x=df_view.index, y=df_view['indicator_value'], name='RSI', line=dict(color='purple')), row=2, col=1)
            # Niveles OB/OS
            ob = indicator_params.get('overbought', 70)
            os_val = indicator_params.get('oversold', 30)
            fig.add_hline(y=ob, line_dash="dash", line_color="red", row=2, col=1)
            fig.add_hline(y=os_val, line_dash="dash", line_color="green", row=2, col=1)
            
        elif indicator_name == 'macd':
            # MACD Line
            fig.add_trace(go.Scatter(x=df_view.index, y=df_view['macd'], name='MACD', line=dict(color='blue')), row=2, col=1)
            # Signal Line
            fig.add_trace(go.Scatter(x=df_view.index, y=df_view['macd_signal_line'], name='Signal', line=dict(color='orange')), row=2, col=1)
            # Histogram
            colors = ['green' if v >= 0 else 'red' for v in df_view['macd'] - df_view['macd_signal_line']]
            fig.add_trace(go.Bar(x=df_view.index, y=df_view['macd'] - df_view['macd_signal_line'], name='Hist', marker_color=colors), row=2, col=1)

    fig.update_layout(title=title, template='plotly_dark', height=900, xaxis_rangeslider_visible=False)
    return fig

In [4]:
# --- Ejecución del Análisis ---

trade_file = 'trades_UNIUSDT_1h_RSI_ob75os20p20_L.csv'

try:
    # 1. Cargar Trades
    print(f"Cargando trades: {trade_file}")
    df_trades = load_trades(trade_file)
    
    # 2. Inferir Metadatos
    ticker = df_trades['ticker'].iloc[0] if 'ticker' in df_trades.columns else 'UNIUSDT'
    parts = trade_file.split('_')
    timeframe = parts[2] if len(parts) > 2 else '1h'
    
    # 3. Inferir Estrategia
    indicator, params = parse_strategy_from_filename(trade_file)
    print(f"Estrategia detectada: {indicator} con params {params}")
    
    # 4. Cargar Mercado
    df_market = load_market_data(ticker, timeframe)
    
    # 5. Visualizar
    fig = plot_trades_on_chart(df_market, df_trades, indicator_name=indicator, indicator_params=params, title=f"Trades: {trade_file}")
    fig.show()
    
except Exception as e:
    print(f"Error durante la ejecución: {e}")
    import traceback
    traceback.print_exc()

Cargando trades: trades_UNIUSDT_1h_RSI_ob75os20p20_L.csv
Estrategia detectada: rsi con params {'overbought': 75, 'oversold': 20, 'period': 20}
Cargando datos de mercado desde: ..\data\market\uniusdt_1h_20200917_20251112.parquet
Calculando indicador: rsi con params {'overbought': 75, 'oversold': 20, 'period': 20}
